# Pendulum swing-up — value iteration vs LQR vs PPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/topics/reinforcement_learning/pendulum_value_iteration_vs_lqr_vs_rl.ipynb)

This notebook compares three ways to synthesize a policy for the **same** swing-up problem and the **same** quadratic cost $J$. All three return a feedback law $u=\pi(x)$ that tries to minimize $J$.

1. **Value iteration (VI)**: global dynamic programming on a grid — the discretized nonlinear optimum.
2. **LQR**: local linear-quadratic feedback at the upright equilibrium.
3. **PPO**: a neural policy trained by reinforcement learning. The rollout environment's reward is $r = -g(x,u,t)\,\Delta t$, so maximizing return is the same as minimizing $J$.

VI is the reference solution of the discretized OCP. LQR is exact only near $\bar x$. PPO is model-free: its update never uses $f$, only sampled trajectories.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox; PPO runs on its native JAX planner.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np

from minilink import (
    DynamicProgrammingPlanner,
    InvertedPendulum,
    LQRPlanner,
    MonteCarloEvaluator,
    PlanningProblem,
    QuadraticCost,
    ReinforcementLearningPlanner,
    StochasticPlanningProblem,
    Uniform,
    compare,
)

## 1. Plant

We load a minilink catalog class (`InvertedPendulum`) that already defines the equations of motion and the state/input labels. The state is $x = [\theta,\;\dot\theta]$ and the input is the pivot torque $u$. The dynamics are
$$\dot x = f(x,u).$$
Here $\theta = 0$ is upright, so the hanging-down position is $x_0 = [-\pi,\; 0]$ and the target is $\bar x = [0,\; 0]$. We also set the **domain** — bounds on $x$ and $|u|\le u_{\max}$ — used later by the grid.


In [ ]:
UPRIGHT = np.array([0.0, 0.0])  # target (upright) and LQR linearization point
X0 = np.array([-np.pi, 0.0])  # hanging down
TORQUE = 1.0
DT = 0.05
TF = 10.0
X_GRID = (201, 201)
U_GRID = (21,)
TOL = 0.1
INF = 500.0
Q = np.diag([1.0 / DT, 0.1 / DT])
R = np.diag([0.1 / DT])

In [ ]:
plant = InvertedPendulum()

# Pendulum parameters
plant.params["m"] = 0.1
plant.params["l"] = 0.5
plant.params["I"] = 1.0 / 12.0 * 0.1 * 1.0**2
plant.params["gravity"] = 9.81
plant.params["d"] = 0.0

# Pendulum bounds
plant.state.lower_bound = np.array([-2.0 * np.pi, -12])
plant.state.upper_bound = np.array([+2.0 * np.pi, +12])
plant.inputs["u"].lower_bound = np.array([-TORQUE])
plant.inputs["u"].upper_bound = np.array([+TORQUE])

# Pendulum initial state
plant.x0 = X0

## 2. Cost function

All three controllers minimize the same quadratic performance metric
$$J = \int_0^{t_f} \big( (x-\bar x)' Q (x-\bar x) + u' R u \big)\, dt.$$
For PPO we will use the equivalent reward $r = -g(x,u)\,\Delta t$, so maximizing return is the same as minimizing $J$.


In [ ]:
cost = QuadraticCost.from_system(plant, xbar=UPRIGHT, Q=Q, R=R)

## 3. Planning problem

A `PlanningProblem` packages the plant, the cost, and the goal. The optimal-control problem is
$$\min_{\pi}\; J \quad\text{s.t.}\quad \dot x = f\big(x,\pi(x)\big),\quad |u|\le u_{\max}.$$
Value iteration solves this on a grid. LQR and PPO target the same $J$ by different approximations.


In [ ]:
problem = PlanningProblem(plant, x_goal=UPRIGHT, cost=cost)

## 4. Value iteration

We discretize $x$ and $u$ on a grid and solve the discrete Bellman equation for the cost-to-go $J^*$:
$$J^*(x) = \min_u \Big\{ g(x,u)\,\Delta t + J^*\big(x + f(x,u)\,\Delta t\big) \Big\}.$$
The minimizing $u$ is the global (discretized) policy $\pi^*(x)$ — the reference solution. Here the state grid is $201\times 201$, the torque has 21 levels, and $\Delta t = 0.05\,\mathrm{s}$.


In [ ]:
planner = DynamicProgrammingPlanner(
    problem,
    x_grid=X_GRID,
    u_grid=U_GRID,
    dt=DT,
    tol=TOL,
    max_iterations=2000,
    out_of_bound_cost=INF,
    verbose=True,
)

sol_vi = planner.solve()
vi_ctl = sol_vi.policy

In [ ]:
planner.plot_cost2go(jmax=INF)

## 5. LQR

Linearize the plant at the upright equilibrium $(\bar x, \bar u)$:
$$\dot{\tilde x} = A\tilde x + B\tilde u,\qquad \tilde x = x-\bar x.$$
The infinite-horizon LQR gain $K$ minimizes the same quadratic $J$ for this linear model:
$$u = \bar u - K(x-\bar x).$$
Same $Q$, $R$, and plant as value iteration — but no torque limits in the synthesis, and no validity away from $\bar x$. `LQRPlanner` linearizes the plant at the cost's target and solves the Riccati equation; its solution carries the law, the closed-loop poles, and the cost-to-go of the linear model.

In [ ]:
sol_lqr = LQRPlanner(problem).solve()
lqr_ctl = sol_lqr.policy
print(sol_lqr.solver)

## 6. PPO

PPO is model-free: its update never uses $f$, only sampled transitions, which minilink simulates in parallel on the compiled plant. The task is the same plant and the same cost, stated as a `StochasticPlanningProblem`: initial states are drawn uniformly over $\theta \in [-\pi, \pi]$ and $\dot\theta \in [-6, 6]$, so the policy must learn swing-up, not only balance, and each episode lasts $t_f$. The reward at each step is
$$r = -g(x,u,t)\,\Delta t,$$
so maximizing expected return is the same as minimizing $J$. The policy sees the state scaled by its bounds rather than periodic angle features: this cost is quadratic in $\theta$ itself, so upright after a full turn, $\theta = 2\pi$, is not the target.

In [ ]:
ppo_problem = StochasticPlanningProblem(
    plant,
    cost=cost,
    tf=np.inf,
    x0_distribution=Uniform([-np.pi, -6.0], [+np.pi, +6.0]),
)

In [ ]:
ppo = ReinforcementLearningPlanner(
    ppo_problem,
    dt=DT,
    episode_length=TF,
    hidden=(64, 64),
    algorithm="ppo",
    n_envs=64,
    n_steps=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
    verbose=False,
)

In [ ]:
sol_ppo = ppo.solve(timesteps=200_000)
print(sol_ppo.solver)
ppo.plot_learning_curve()

ppo_ctl = sol_ppo.policy

## 7. Control laws

The three maps $u=\pi(\theta,\dot\theta)$ on the same state box and the same torque scale. A well-trained PPO policy should resemble VI in the region visited during training. LQR is the linear plane through $\bar x$. `compare` names the three solutions; `print` reads their solver records side by side.

In [ ]:
race = compare(VI=sol_vi, LQR=sol_lqr, PPO=sol_ppo)
print(race)

In [ ]:
race.plot_control_law()

Each method's own cost-to-go, on one colour scale: the value-iteration table, and the Riccati form $(x-\bar x)^T S (x-\bar x)$ of the linear model, which is exact only where the linear model is. PPO's critic is not drawn: it estimates the discounted return of its own $\gamma$, not this $J$.

In [ ]:
race.plot_cost_to_go(jmax=INF)

## 8. Closed-loop simulation

We wire each policy as state feedback $u=\pi(x)$ and integrate from the hanging position $x_0 = [-\pi,\; 0]$.


In [ ]:
closed = []
for ctl, name in (
    (vi_ctl, "Pendulum with VI"),
    (lqr_ctl, "Pendulum with LQR"),
    (ppo_ctl, "Pendulum with PPO"),
):
    plant.x0 = X0
    cl = ctl @ plant
    cl.name = name
    cl.plot_diagram()
    traj = cl.compute_trajectory(tf=TF, n_steps=int(TF / DT) + 1, solver="euler")
    cl.plot_trajectory(traj)
    closed.append((cl, traj))

(cl_vi, traj_vi), (cl_lqr, traj_lqr), (cl_ppo, traj_ppo) = closed

## 9. Animation — VI

Closed-loop motion under the value-iteration policy, from hanging down.


In [ ]:
cl_vi.animate(traj_vi)

## 9. Animation — LQR

Same initial state under LQR. Compare the torque and the path to VI and PPO.


In [ ]:
cl_lqr.animate(traj_lqr)

## 9. Animation — PPO

Same initial state under the trained neural policy.


In [ ]:
cl_ppo.animate(traj_ppo)

## 10. Performance

The same $J$ evaluated along each closed-loop trajectory. VI is the global optimum of the discretized problem. A well-trained PPO policy should approach that $J$; LQR typically spends more torque from the hanging position.


In [ ]:
# Each pendulum as it ran inside its loop: the state and the torque it received
cl_vi.plot_cost(cost, of=plant, traj=traj_vi)
cl_lqr.plot_cost(cost, of=plant, traj=traj_lqr)
cl_ppo.plot_cost(cost, of=plant, traj=traj_ppo)

## 11. One yardstick: Monte Carlo over many starts

Section 10 compared the three laws from one start. `MonteCarloEvaluator` scores each of them on the same 100 draws of the start distribution PPO trained on, with the same held-input simulation and the same $J$, and reports the spread, the worst trial, and the failure rate (trials that leave the state box). VI and LQR are just controller blocks here, like the neural policy: `race.evaluate` runs the one evaluator on every solution and fills the table's last column.

In [ ]:
evaluator = MonteCarloEvaluator(
    ppo_problem, dt=DT, n_trials=100, episode_length=TF, backend="numpy", seed=1
)
print(race.evaluate(evaluator))